In [50]:
![https://deeprevision.github.io/posts/001-transformer/transformer.png](https://deeprevision.github.io/posts/001-transformer/transformer.png)

The filename, directory name, or volume label syntax is incorrect.


In [4]:
text1= "The cat  chased the dog"
text2= "The dog  chased the cat"

text = "the capital of France is "

In [52]:
def tokenizer(text):
    return text.split() #en basit şekliyle kelimelere bölme işlemi yapar yani tokinize eder

In [53]:
tokenizer(text1)

['The', 'cat', 'chased', 'the', 'dog']

In [67]:
#vocab = {
#    "The": 0,
#    "cat": 1,
#    "dog": 2,
#    "chased": 3,
#    "capital": 4,
#    "of": 5,
#    "France": 6,
#    "is": 7,
#    "<unk>": 8,
#    "the": 9
#}                                 #sözlük oluşturduk ve her kelimeye bir sayı atadık 

import json
with open("tokenizer.json", "r") as f:
    vocab = json.load(f) #json dosyasını okuyarak vocab sözlüğünü oluşturduk
    
def tokenizer2(text):
    parts = text.split()
    ids =[]
    for part in parts:
        if part in vocab:
            value = vocab[part]
        else:
            value = vocab["<unk>"]
        ids.append(value)
    return ids
tokenids1= tokenizer2(text1)
tokenids1



[0, 1, 3, 9, 2]

In [55]:
tokenizer2("How are you?")

[8, 8, 8]

In [56]:
#reverse_vocab = {v:k for k,v in vocab.items()} #sözlüğün tersini oluşturduk
reverse_vocab = {id: part for part,id in vocab.items()} 
#sözlüğün tersini oluşturduk
reverse_vocab

{0: 'The',
 1: 'cat',
 2: 'dog',
 3: 'chased',
 4: 'capital',
 5: 'of',
 6: 'France',
 7: 'is',
 8: '<unk>',
 9: 'the'}

In [57]:
def detokenizer(ids):
    text = ""
    for id in ids:
        part = reverse_vocab[id]
        text += part + " "
    text = text.strip() #sonundaki boşluğu kaldırır
    return text

detokenizer(tokenids1)

'The cat chased the dog'

In [58]:
##sistemden kontrol ettiğimizde tiktokenizer.vercel.app/ adresine giderek tokenizer ve detokenizer işlemlerini yapabiliriz.
##burada da gördükki gpt içinde büyük the için ve küçük the için farklı tokenler var yani büyük harf ve küçük harf ayrımı var. bu yüzden büyük harf ve küçük harf ayrımını yaparak tokenizasyon işlemi yapmamız gerekiyor. bu yüzden tokenizer2 fonksiyonunu oluşturduk ve burada büyük harf ve küçük harf ayrımını yaparak tokenizasyon işlemi yapıyoruz. eğer kelime sözlükte yoksa <unk> tokenini kullanıyoruz. detokenizer fonksiyonunda ise token idlerini alarak tekrar kelimelere çeviriyoruz.
##onların sözlülerinde nerede olduğunu bilmiş olduk
##arxiv org hocanın makalesini bakabliriz


In [59]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
enc.encode(text1), enc.decode(tokenids1)

([464, 3797, 220, 26172, 262, 3290], '!"$*#')

In [60]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
gpt2_ids = enc.encode(text1)
gpt2_ids

[464, 3797, 220, 26172, 262, 3290]

In [61]:
enc.decode(gpt2_ids), enc.decode(tokenids1)

('The cat  chased the dog', '!"$*#')

In [62]:
enc.decode([333,777])

'ur these'

In [63]:
#github.com/openai/tiktoken adresine giderek tiktoken kütüphanesini inceleyebiliriz. burada gpt2 tokenizerını kullanarak text1'i token idlerine çevirdik ve sonra bu token idlerini decode ederek tekrar text1'e çevirdik. ayrıca kendi oluşturduğumuz tokenizer2 fonksiyonunu kullanarak text1'i token idlerine çevirdik ve sonra bu token idlerini decode ederek tekrar text1'e çevirdik. gördüğümüz gibi gpt2 tokenizerı ile oluşturduğumuz tokenizer2 fonksiyonu farklı token idleri üretiyor çünkü gpt2 tokenizerı daha karmaşık bir tokenizasyon işlemi yapıyor.

In [64]:
enc = tiktoken.get_encoding("o200k_base")
enc.n_vocab

200019

In [65]:
o200_ids = enc.encode(text1)
o200_ids

[976, 9059, 220, 135896, 290, 6446]

In [ ]:
#burada birden fazla gemma gpt4 gibi kullanbiliriz dışarıdan 
#load model 
from transformers import AutoTokenizer, AutoModelForCausalLM
processor = AutoTokenizer.from_pretrained("gemma-2b-it")
model = AutoModelForCausalLM.from_pretrained("gemma-2b-it")
#hata veriyor huggingface izin vermiyor-proccessor ve model yüklenemiyor


c:\Users\ceren\Desktop\llm_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


OSError: gemma-2b-it is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
gemma_ids = processor.tokenizer.encode(text1)
gemma_ids

In [ ]:
#sözlük size nedir
processor.tokenizer.vocab_size

In [ ]:
#tekrar metne çevirme
processor.tokenizer.decode(gemma_ids)
#processor.tokenizer.decode(gemma_ids)[5:]

In [ ]:
#json dosyasını incelemek istersekte
processor.tokenizer.save_vocabulary("gemma_tokenizer.json")

In [ ]:
#diğer bir yöntemile 
processor.tokenizer.get_vocab()

In [ ]:
import json
with open("gemma_tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(processor.tokenizer.get_vocab(), f)
    
#biz bir modelin ne oldugunu neye karşılık geldiğini görebiliriz gemmada ne sözlük varmış diye

In [3]:
from tokenizer import Tokenizer

In [6]:
tokenizer = Tokenizer("tokenizer.json")
tokenizer.encode("states"), tokenizer.decode([4, 58])

([4, 58], 'states')

In [7]:
#komple tüm metini tokenleme
with open("text.txt", "r") as f:
    text = f.read()
    
text


'the capital of the united states is not london. the capital of france is paris, and berlin is the capital of germany. rome is in italy, \n\nmadrid is in spain, and lisbon is in portugal. the capital of the united kingdom is not paris, and the capital of the united states is not berlin. \nalthough these places are often mentioned together, although these capitals are often mentioned together, although these are often mentioned together, \neach country has its own capital, and each country has its own city, and each capital has its own identity, and each capital has its own history. washington \nis the capital of the united states, and london is the capital of the united kingdom. paris is known for art and fashion, and berlin is known for art and \nhistory, and rome is known for art and history, and madrid is known for culture and history, and lisbon is known for culture and art. rome is rich with culture, \nrome is rich with history, rome is rich with art, and madrid is rich with art a

In [8]:
tokens= tokenizer.encode(text)
tokens

[0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 4,
 58,
 61,
 5,
 61,
 6,
 61,
 7,
 59,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 8,
 61,
 5,
 61,
 9,
 60,
 61,
 10,
 61,
 11,
 61,
 5,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 12,
 59,
 61,
 13,
 61,
 5,
 61,
 14,
 61,
 15,
 60,
 61,
 16,
 61,
 5,
 61,
 14,
 61,
 17,
 60,
 61,
 10,
 61,
 18,
 61,
 5,
 61,
 14,
 61,
 19,
 59,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 20,
 61,
 5,
 61,
 6,
 61,
 9,
 60,
 61,
 10,
 61,
 0,
 61,
 1,
 61,
 2,
 61,
 0,
 61,
 3,
 61,
 4,
 58,
 61,
 5,
 61,
 6,
 61,
 11,
 59,
 61,
 22,
 61,
 23,
 61,
 24,
 58,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 22,
 61,
 23,
 61,
 1,
 58,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 22,
 61,
 23,
 61,
 25,
 61,
 26,
 61,
 27,
 57,
 61,
 28,
 60,
 61,
 29,
 61,
 30,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 1,
 60,
 61,
 10,
 61,
 29,
 61,
 30,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 37,
 60,
 61,
 10,
 61,
 29,
 61,
 1,
 61,
 31,
 61,
 32,
 61,
 33,
 61,
 34,
 60,
 

In [1]:
%pip install sentencepiece -qU

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sentencepiece as spm
spm.SentencePieceTrainer.Train(
    input="text.txt",
    model_prefix="spm_tokenizer", 
    vocab_size=64, 
    model_type="bpe"   #bype pair ?
)


In [5]:
spm_tokenizer = spm.SentencePieceProcessor(model_file="spm_tokenizer.model")

spm_ids=spm_tokenizer.encode(text1)
spm_tokens = spm_tokenizer.encode(text1, out_type=str)
spm_ids,spm_tokens

([39, 0, 48, 43, 7, 41, 40, 7, 48, 41, 46, 30, 9, 39, 51, 45, 60],
 ['▁',
  'T',
  'h',
  'e',
  '▁c',
  'a',
  't',
  '▁c',
  'h',
  'a',
  's',
  'ed',
  '▁the',
  '▁',
  'd',
  'o',
  'g'])

In [6]:
%pip install tokenizers -qU

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE   
from tokenizers.trainers import BpeTrainer  #bizim için tokenleyen bpe trainer
from tokenizers.pre_tokenizers import Whitespace

In [8]:
hf_tokenizer = Tokenizer(BPE())

In [10]:
hf_tokenizer.get_vocab_size()

0

In [11]:
hf_tokenizer.pre_tokenizer = Whitespace()

In [12]:
trainer = BpeTrainer(vocab_size=64, special_tokens=["<unk>"])
hf_tokenizer.train(["text.txt"], trainer)

In [13]:
hf_tokenizer.get_vocab_size(),hf_tokenizer.encode(text1).ids

(64, [10, 7, 5, 3, 21, 5, 10, 49, 44, 28, 6, 16, 9])

In [14]:
hf_tokenizer.save("hf_tokenizer.json")

In [15]:
from transformers import PreTrainedTokenizerFast

fast_tokenizer = PreTrainedTokenizerFast(tokenizer_file="hf_tokenizer.json")

fast_tokenizer.encode(text1)

[10, 7, 5, 3, 21, 5, 10, 49, 44, 28, 6, 16, 9]

In [ ]:
#cli login to hugginface
#huggingface-cli login 

#terminelde olacak ve hugging face acces token oluşturmalısın


In [ ]:
#fast_tokenizer.push_to_hub("my-hf-tokenizer")

In [2]:
%pip install transformers torch tiktokenize datasets matplotlib -Uq

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for tiktoken (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [49 lines of output]
      C:\Users\ceren\AppData\Local\Temp\pip-build-env-jjegbgkr\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ************************************************